# Roteiro de Aula: Feature Engineering com banco de dados bancários

Este notebook orienta a prática de Feature Engineering com o arquivo `bank-vf.csv`.
O foco é analisar o dataset, tratar valores faltantes, codificar variáveis categóricas, escalar variáveis numéricas, discretizar, criar novos atributos e selecionar features.

## Objetivos
- Explorar a base `bank-vf.csv` e compreender o significado das variáveis.
- Aplicar tratamento de dados nulos e comparar abordagens.
- Codificar variáveis categóricas usando ordinais e One-Hot.
- Comparar escalonamentos: MinMax, Standard e Robust.
- Discretizar uma variável numérica e interpretar os resultados.
- Criar novos atributos e avaliar sua utilidade.
- Realizar amostragem em base desbalanceada e selecionar features relevantes.

## 1. Carregamento e inspeção inicial
1. Importe `pandas`, `numpy`, `seaborn` e `matplotlib.pyplot`.
2. Carregue `bank-vf.csv` em um DataFrame `df`.
3. Remova a coluna `Unnamed: 0` se ela existir.
4. Exiba `df.head()` e `df.info()`.

### Perguntas
- Quantas linhas e colunas existem no dataset?
- Qual é a variável alvo?
- Quais colunas parecem conter valores faltantes?

In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Carregue a base de dados
df = pd.read_csv('C:\\Users\\negri\\OneDrive\\Documentos\\GitHub\\Hashtag_AnaliseDados\\Python\\Cursos\\PUC Ciência de Dados\\Projetos\\Machine Learning\\03_2 Aula\\bank-vf.csv')
# Remova a coluna de índice se existir
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

# Inspeção inicial
print(df.shape)
display(df.head())
print('Informações do DataFrame:')
print(df.info())

(45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,NaN,management,NaN,tertiary,no,2143.0,yes,no,NaN,5.0,may,NaN,1.0,-1.0,0.0,NaN,no
1,44.0,technician,single,secondary,no,29.0,yes,no,NaN,5.0,may,151.0,1.0,-1.0,0.0,NaN,no
2,NaN,entrepreneur,married,secondary,no,2.0,yes,yes,NaN,5.0,may,NaN,1.0,-1.0,0.0,NaN,no
3,47.0,blue-collar,married,NaN,no,1506.0,yes,no,NaN,5.0,may,92.0,1.0,-1.0,0.0,NaN,no
4,NaN,NaN,single,NaN,no,1.0,no,no,NaN,5.0,may,198.0,1.0,NaN,0.0,NaN,no


Informações do DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        29630 non-null  float64
 1   job        44923 non-null  str    
 2   marital    24972 non-null  str    
 3   education  43354 non-null  str    
 4   default    45211 non-null  str    
 5   balance    29575 non-null  float64
 6   housing    45211 non-null  str    
 7   loan       45211 non-null  str    
 8   contact    32191 non-null  str    
 9   day        45211 non-null  float64
 10  month      45211 non-null  str    
 11  duration   26920 non-null  float64
 12  campaign   45211 non-null  float64
 13  pdays      35436 non-null  float64
 14  previous   38971 non-null  float64
 15  poutcome   8252 non-null   str    
 16  y          45211 non-null  str    
dtypes: float64(7), str(10)
memory usage: 5.9 MB
None


## 2. Divisão treino/teste estratificada
1. Divida a base em treino e teste antes de qualquer tratamento de dados.
2. Use `train_test_split(..., stratify=df['y'], random_state=42)` para preservar a proporção da variável resposta.
3. Mantenha todo o pré-processamento e engenharia de features apenas no conjunto de treino.
4. Reserve `df_test` como conjunto de validação final e não o utilize durante o ajuste de imputadores, encoders ou scalers.

### Observação
- A partir desta etapa, `df` representa o conjunto de treino (`df_train`).
- `df_test` deve ser usado apenas para avaliação final após todas as transformações serem definidas.

In [6]:
from sklearn.model_selection import train_test_split

# Divisão estratificada pela variável resposta
df_train, df_test = train_test_split(df, test_size=0.2, stratify=df['y'], random_state=42)
print('Train shape:', df_train.shape)
print('Test shape :', df_test.shape)

# A partir daqui, usamos apenas o conjunto de treino para pré-processamento
df = df_train.copy()

Train shape: (36168, 17)
Test shape : (9043, 17)


## 3. EDA automatizada com ydata-profiling
1. Instale `ydata-profiling` e gere um relatório exploratório automatizado.
2. Verifique alertas de dados faltantes, variáveis constantes, correlações altas e possíveis outliers.
3. Revise as estatísticas de cada coluna antes de aplicar imputação e codificação.
4. Use o relatório para orientar a seleção de técnicas de pré-processamento.

### Exemplo de código
```python
# !pip install ydata-profiling
from ydata_profiling import ProfileReport
profile = ProfileReport(df_train, title='Relatório EDA - bank-vf', explorative=True, minimal=True)
profile.to_file('relatorio_eda_bank_vf.html')
profile.to_notebook_iframe()
```

In [7]:
!pip install ydata-profiling
from ydata_profiling import ProfileReport
profile = ProfileReport(df_train, title='Relatório EDA - bank-vf', explorative=True, minimal=True)
profile.to_file('relatorio_eda_bank_vf.html')
profile.to_notebook_iframe()

ImportError: scipy._cyutility does not export expected C function slice_memviewslice

## 4. Análise de valores faltantes
1. Verifique a quantidade de nulos em cada coluna com `df_train.isnull().sum()`.
2. Visualize a estrutura de nulos usando `missingno` ou gráficos simples.
3. Decida quais colunas precisam de tratamento no conjunto de treino.

### Perguntas
- Quais colunas têm mais valores faltantes?
- É adequado usar imputação por mediana em todas as colunas numéricas?
- Como tratar colunas categóricas com valores faltantes?

In [ ]:
# Inspeção dos valores faltantes
missing_counts = df_train.isnull().sum().sort_values(ascending=False)
display(missing_counts)

# Visualizar nulos com missingno (opcional)
# import missingno as msno
# msno.matrix(df_train)
# plt.show()

## 5. Tratamento de valores ausentes
1. Crie duas cópias do DataFrame: `df_simple` e `df_knn`.
2. Em `df_simple`, use `SimpleImputer` para preencher numéricos com a mediana e categóricos com a moda.
3. Em `df_knn`, preencha categorias com moda, transforme em dummy e aplique `KNNImputer`.
4. Compare a distribuição de uma variável numérica como `age` ou `balance` entre as abordagens.

### Perguntas
- Qual método preserva melhor a distribuição original?
- Quais são as limitações do `KNNImputer`?

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

# Preparar os DataFrames de comparacao
df_simple = df.copy()
df_knn = df.copy()

# Identifique colunas numericas e categoricas
col_num = df.select_dtypes(include=np.number).columns
col_cat = df.select_dtypes(include=['object', 'category']).columns

# TODO: aplique SimpleImputer em df_simple
# TODO: aplique KNNImputer em df_knn, depois copie os valores de volta

print('df_simple e df_knn prontos para comparação.')

## 6. Encoding de variáveis categóricas
1. Identifique as colunas categóricas e determine quais podem receber `OrdinalEncoder`.
2. Use `OneHotEncoder` para as demais variáveis.
3. Crie um `ColumnTransformer` que mantenha colunas numéricas e transforme as categóricas.
4. Exporte um DataFrame codificado e compare formas de codificação.

### Perguntas
- Qual a diferença entre ordinal e one-hot encoding?
- Quando o `OrdinalEncoder` pode introduzir ordens falsas?

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# Exemplo de colunas a codificar
cols_ordinais = ['education']
cols_nominais = [col for col in df.select_dtypes(include=['object']).columns if col not in cols_ordinais and col != 'y']
cols_numericas = df.select_dtypes(include=np.number).columns.tolist()

education_order = ['primary', 'secondary', 'tertiary']

preprocessor = ColumnTransformer(transformers=[
    ('ord', OrdinalEncoder(categories=[education_order], handle_unknown='use_encoded_value', unknown_value=-1), cols_ordinais),
    ('ohe', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), cols_nominais)
], remainder='passthrough')

# TODO: aplique fit_transform no DataFrame de treino
# TODO: construa um DataFrame com nomes de colunas legíveis

## 7. Escalonamento de dados numéricos
1. Use `MinMaxScaler`, `StandardScaler` e `RobustScaler` em um mesmo conjunto numérico.
2. Compare a distribuição de pelo menos uma variável após cada scaler.
3. Registre as vantagens e desvantagens de cada método.

### Perguntas
- Qual scaler é mais sensível a outliers?
- Para que tipo de modelo cada scaler é recomendado?

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# TODO: use as três técnicas de escalonamento nas colunas numéricas
# df_minmax = ...
# df_standard = ...
# df_robust = ...

print('Prepare as comparações para a variável balance ou duration.')

## 8. Discretização de variáveis numéricas
1. Escolha a variável `duration` ou `balance` para discretizar.
2. Aplique `KBinsDiscretizer` com estratégias `uniform`, `quantile` e `kmeans`.
3. Analise como cada método agrupa os valores.

### Perguntas
- Qual estratégia é mais útil quando há muitos valores extremos?
- A discretização ajuda a reduzir o impacto de outliers?

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

# TODO: selecione uma coluna e aplique as três estratégias
# discretizer = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='uniform')
# ...

print('Compare os resultados de discretização entre as 3 estratégias.')

## 9. Criação de novos atributos
1. Crie um novo atributo a partir de duas variáveis numéricas, por exemplo `balance / (duration + 1)` ou `campaign * duration`.
2. Explique a hipótese de negócio por trás desse novo atributo.
3. Analise a correlação do novo atributo com a variável alvo.

### Perguntas
- Sua nova feature é informativa?
- Há risco de vazamento de dados? Por quê?

In [ ]:
# TODO: crie um novo atributo de interação ou transformação
# df['balance_duration_ratio'] = (df['balance'] + 1) / (df['duration'] + 1)
# df[['balance_duration_ratio']].describe()

print('Defina um novo atributo e justifique sua criação.')

## 10. Amostragem em base desbalanceada
1. Verifique o balanceamento da variável alvo `y`.
2. Use `RandomUnderSampler` e `SMOTE` para gerar bases balanceadas.
3. Compare o tamanho e a distribuição de cada base.

### Perguntas
- Qual técnica preserva mais informação?
- Quando é melhor usar undersampling em vez de oversampling?

In [ ]:
# TODO: verifique a proporção de classes em y
# y_numeric = df['y'].map({'no': 0, 'yes': 1})
# TODO: use RandomUnderSampler e SMOTE

print('Compare a distribuição original e as versões balanceadas.')

## 11. Seleção de features
1. Aplique `SelectKBest` ou `chi2` para seleção filtro.
2. Aplique `RFE` para seleção wrapper.
3. Aplique `SelectFromModel` com `RandomForestClassifier` ou `LogisticRegression` para seleção embarcada.
4. Compare as listas de features selecionadas.

### Perguntas
- Quais features aparecem em mais de um método?
- O método escolhido depende do tipo de modelo?

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, RFE, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# TODO: aplique os três métodos de seleção 
# selector_kbest = SelectKBest(f_classif, k=10)
# selector_rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=10)
# selector_sfm = SelectFromModel(RandomForestClassifier(random_state=42), max_features=10)

print('Compare as features selecionadas por cada método.')

## 12. Conclusão
Descreva brevemente as escolhas feitas durante o processo de Feature Engineering e o que você aprendeu com a comparação de técnicas.